# Gold set diagnostic: independence of the evaluation

The verified gold set contains 217,021 boxes, of which ~192,668 originated as
teacher pre-annotations that the annotator left unchanged and ~24,342 were
added by hand. Baselines were trained on teacher-generated labels, so the
first group is not independent of the models being evaluated. This notebook
scores the two groups separately.

It also handles the 15 tiles with no ground-truth boxes, for which average
precision is undefined; these are reported as a false-positive rate instead.

Needs in Drive: `gold_predictions.json`, `gold_v2_verified.zip`,
`gold_v2_coco_c30.json`, `test_classes.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q numpy pandas
import os, json, glob, re, csv
import numpy as np, pandas as pd
from collections import defaultdict, Counter

preds_all = json.load(open(f'{DRIVE}/gold_predictions.json'))
print('models:', list(preds_all))

!rm -rf /content/gold_ann
!unzip -q {DRIVE}/gold_v2_verified.zip -d /content/gold_ann
ann_path = glob.glob('/content/gold_ann/**/instances_default.json', recursive=True)[0]
verified = json.load(open(ann_path))
teacher  = json.load(open(f'{DRIVE}/gold_v2_coco_c30.json'))
print('verified boxes:', len(verified['annotations']),
      '| teacher boxes:', len(teacher['annotations']))

In [ ]:
def to_xyxy(d):
    names = {im['id']: im['file_name'].split('/')[-1] for im in d['images']}
    out = defaultdict(list)
    for a in d['annotations']:
        x, y, w, h = a['bbox']
        out[names[a['image_id']]].append([x, y, x+w, y+h])
    for n in names.values():
        out.setdefault(n, [])
    return dict(out)

gt_all  = to_xyxy(verified)
gt_teach = to_xyxy(teacher)

def iou_matrix(A, B):
    if len(A) == 0 or len(B) == 0:
        return np.zeros((len(A), len(B)), np.float32)
    A = np.asarray(A, np.float32); B = np.asarray(B, np.float32)
    ix1 = np.maximum(A[:,0,None], B[None,:,0]); iy1 = np.maximum(A[:,1,None], B[None,:,1])
    ix2 = np.minimum(A[:,2,None], B[None,:,2]); iy2 = np.minimum(A[:,3,None], B[None,:,3])
    inter = np.clip(ix2-ix1,0,None) * np.clip(iy2-iy1,0,None)
    aA = ((A[:,2]-A[:,0])*(A[:,3]-A[:,1]))[:,None]
    aB = ((B[:,2]-B[:,0])*(B[:,3]-B[:,1]))[None,:]
    u = aA + aB - inter
    return np.divide(inter, u, out=np.zeros_like(inter), where=u>0)

# Split verified boxes: those matching a teacher proposal vs those added by hand
MATCH = 0.5
gt_kept, gt_added = {}, {}
n_k = n_a = 0
for t, boxes in gt_all.items():
    tb = gt_teach.get(t, [])
    if not boxes:
        gt_kept[t], gt_added[t] = [], []
        continue
    M = iou_matrix(boxes, tb)
    matched = M.max(1) >= MATCH if M.shape[1] else np.zeros(len(boxes), bool)
    gt_kept[t]  = [b for b, m in zip(boxes, matched) if m]
    gt_added[t] = [b for b, m in zip(boxes, matched) if not m]
    n_k += int(matched.sum()); n_a += int((~matched).sum())
print(f'teacher-derived: {n_k:,} ({n_k/(n_k+n_a)*100:.1f}%) | '
      f'human-added: {n_a:,} ({n_a/(n_k+n_a)*100:.1f}%)')

In [ ]:
def ap_at(preds, gtd, thr, keys):
    rows, n_gt = [], 0
    for t in keys:
        g = gtd.get(t, []); p = sorted(preds.get(t, []), key=lambda b: -b[4])
        n_gt += len(g)
        if not p: continue
        M = iou_matrix([b[:4] for b in p], g)
        taken = set()
        for i, b in enumerate(p):
            j = -1
            if M.shape[1]:
                for cand in np.argsort(-M[i]):
                    if M[i, cand] < thr: break
                    if cand not in taken:
                        j = int(cand); break
            if j >= 0: taken.add(j); rows.append((b[4], 1))
            else:      rows.append((b[4], 0))
    if n_gt == 0 or not rows: return None
    rows.sort(key=lambda r: -r[0])
    tp = np.cumsum([r[1] for r in rows]); fp = np.cumsum([1-r[1] for r in rows])
    rec = tp/n_gt; prec = tp/np.maximum(tp+fp, 1e-9)
    mprec = np.concatenate([[0], prec, [0]])
    for i in range(len(mprec)-2, -1, -1):
        mprec[i] = max(mprec[i], mprec[i+1])
    ap = float(np.mean(np.interp(np.linspace(0,1,101),
                                 np.concatenate([[0],rec,[1]]), mprec[:len(rec)+2])))
    return ap, float(rec[-1])

def score(preds, gtd, keys):
    r = ap_at(preds, gtd, 0.5, keys)
    if r is None: return None
    ap50, rec = r
    aps = [ap_at(preds, gtd, t, keys)[0] for t in np.arange(0.5, 1.0, 0.05)]
    return dict(mAP50=round(ap50,4), mAP50_95=round(float(np.mean(aps)),4),
                recall=round(rec,4))

## Teacher-derived boxes versus boxes added by the annotator

If the models were merely reproducing their teacher, performance on the
human-added subset would fall sharply relative to the teacher-derived subset.

In [ ]:
nonempty = [t for t in gt_all if gt_all[t]]
rows = []
for name, preds in preds_all.items():
    for label, gtd in [('teacher-derived', gt_kept), ('human-added', gt_added)]:
        keys = [t for t in nonempty if gtd[t]]
        s = score(preds, gtd, keys)
        rows.append(dict(model=name, subset=label, n_boxes=sum(len(gtd[t]) for t in keys), **s))
split = pd.DataFrame(rows)
print(split.to_string(index=False))
split.to_csv(f'{DRIVE}/results_independence.csv', index=False)

## Background tiles

Average precision is undefined where there is no ground truth, so the 15
tiles with no verified boxes are reported as false positives per tile at
several confidence thresholds.

In [ ]:
empty = [t for t in gt_all if not gt_all[t]]
print(f'{len(empty)} tiles with no verified boxes')
rows = []
for name, preds in preds_all.items():
    r = dict(model=name)
    for thr in [0.10, 0.25, 0.50]:
        fp = [sum(1 for b in preds.get(t, []) if b[4] >= thr) for t in empty]
        r[f'FP@{thr:.2f}'] = round(float(np.mean(fp)), 2)
    rows.append(r)
bg = pd.DataFrame(rows)
print(bg.to_string(index=False))
bg.to_csv(f'{DRIVE}/results_background.csv', index=False)

## Teacher scored on the gold set

The annotator model evaluated on the same verified data, for reference. A
student exceeding its teacher on the human-added subset is evidence of
genuine generalisation rather than imitation.

In [ ]:
!pip install -q ultralytics
from ultralytics import YOLO
from tqdm import tqdm
!rm -rf /content/gold
!unzip -q {DRIVE}/gold_v2_tiles.zip -d /content/gold
tiles = sorted(set(glob.glob('/content/gold/gold_v2/*.jpg') +
                   glob.glob('/content/gold/gold_v2/*.JPG')))

tm = YOLO(f'{DRIVE}/teacher_11s_best.pt')
tp = {}
for i in tqdm(range(0, len(tiles), 64)):
    batch = tiles[i:i+64]
    res = tm.predict(batch, imgsz=640, conf=0.001, iou=0.7, max_det=1000, verbose=False)
    for path, r in zip(batch, res):
        b = r.boxes
        tp[os.path.basename(path)] = [
            [float(x1),float(y1),float(x2),float(y2),float(c)]
            for (x1,y1,x2,y2), c in zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy())]

rows = []
for label, gtd in [('all', gt_all), ('teacher-derived', gt_kept), ('human-added', gt_added)]:
    keys = [t for t in nonempty if gtd[t]]
    rows.append(dict(model='teacher_11s', subset=label, **score(tp, gtd, keys)))
tt = pd.DataFrame(rows)
print(tt.to_string(index=False))
tt.to_csv(f'{DRIVE}/results_teacher.csv', index=False)